# Winner Reproduce, Controle

Este notebook reproduz exatamente o pipeline winner (0.84027) para SPR 2026, classificação BI-RADS de laudos de mamografia.

Sem AWP, sem Optuna, sem label smoothing. Thresholds e temperatura fixos no valor do winner.

Se este notebook não atingir aproximadamente 0.84, existe bug no pipeline.

In [ ]:
import os, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import re
import warnings
warnings.filterwarnings('ignore')

MODEL_PATH = '/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1'
TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

MAX_LEN      = 512
BATCH_SIZE   = 8
EPOCHS       = 5
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS      = 5
SEED         = 42
FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.25
NUM_CLASSES  = 7

BEST_TEMP       = 0.958
BEST_THRESHOLDS = [0.95, 1.6, 1.35, 1.0, 0.4, 1.15, 0.1]

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print('Train:', train_df.shape, 'Test:', test_df.shape)

train_texts  = [build_input_text(t) for t in train_df['report'].fillna('').tolist()]
train_labels = train_df['target'].tolist()
test_texts   = [build_input_text(t) for t in test_df['report'].fillna('').tolist()]
print('Exemplo train_text[0]:')
print(train_texts[0][:300])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

class SPRDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
            return_token_type_ids=True,
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'token_type_ids': enc['token_type_ids'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class FocalLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        with autocast():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            loss = loss_fn(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

@torch.no_grad()
def predict(model, loader, return_labels=False):
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        with autocast():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        probs = F.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        if return_labels and 'labels' in batch:
            all_labels.append(batch['labels'].numpy())
    probs = np.concatenate(all_probs, axis=0)
    if return_labels:
        labels = np.concatenate(all_labels, axis=0) if all_labels else None
        return probs, labels
    return probs

def compute_f1(probs, labels):
    preds = probs.argmax(axis=1)
    return f1_score(labels, preds, average='macro')

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
labels_arr = np.array(train_labels)

oof_probs  = np.zeros((len(train_texts), NUM_CLASSES), dtype=np.float32)
test_probs = np.zeros((len(test_texts),  NUM_CLASSES), dtype=np.float32)

test_dataset = SPRDataset(test_texts, labels=None, tokenizer=tokenizer, max_len=MAX_LEN)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_texts, labels_arr)):
    print(f'\n===== Fold {fold+1}/{N_FOLDS} =====')
    tr_texts  = [train_texts[i]  for i in tr_idx]
    tr_labels = [train_labels[i] for i in tr_idx]
    va_texts  = [train_texts[i]  for i in va_idx]
    va_labels = [train_labels[i] for i in va_idx]

    tr_ds = SPRDataset(tr_texts, tr_labels, tokenizer, MAX_LEN)
    va_ds = SPRDataset(va_texts, va_labels, tokenizer, MAX_LEN)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=NUM_CLASSES, local_files_only=True
    ).to(device)

    no_decay = ['bias', 'LayerNorm.weight']
    grouped = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(grouped, lr=LR)
    total_steps  = len(tr_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    loss_fn = FocalLoss()
    scaler = GradScaler()

    best_f1 = -1.0
    best_val_probs  = None
    best_test_probs = None

    for epoch in range(EPOCHS):
        tr_loss = train_epoch(model, tr_loader, optimizer, scheduler, loss_fn, scaler)
        val_probs, val_labels = predict(model, va_loader, return_labels=True)
        val_f1 = compute_f1(val_probs, val_labels)
        print(f'Fold {fold+1} epoch {epoch+1}: train_loss={tr_loss:.4f}  val_f1={val_f1:.5f}')
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_val_probs  = val_probs
            best_test_probs = predict(model, test_loader)

    oof_probs[va_idx] = best_val_probs
    test_probs       += best_test_probs / N_FOLDS
    print(f'Fold {fold+1} best val_f1: {best_f1:.5f}')

    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

oof_argmax_f1 = f1_score(labels_arr, oof_probs.argmax(axis=1), average='macro')
print(f'\nOOF F1 (argmax, sem calibração): {oof_argmax_f1:.5f}')

In [ ]:
def temperature_scale(probs, temp):
    logits = np.log(probs + 1e-10)
    scaled = logits / temp
    exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_s / exp_s.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)

oof_cal   = temperature_scale(oof_probs, BEST_TEMP)
oof_preds = apply_thresholds(oof_cal, BEST_THRESHOLDS)
oof_f1    = f1_score(labels_arr, oof_preds, average='macro')
print(f'OOF F1 (calibração fixa do winner): {oof_f1:.5f}')

test_cal   = temperature_scale(test_probs, BEST_TEMP)
test_preds = apply_thresholds(test_cal, BEST_THRESHOLDS)

In [ ]:
submission = pd.DataFrame({'ID': test_df['ID'], 'target': test_preds})
submission.to_csv('submission.csv', index=False)
print('submission.csv salvo com shape:', submission.shape)
print(submission['target'].value_counts().sort_index())